### Descripcion del Proyecto Orientado

La compa??a de seguros Sure Tomorrow quiere resolver varias tareas con la ayuda de machine learning y te pide que eval?es esa posibilidad.

- Tarea 1: encontrar clientes que sean similares a un cliente determinado. Esto ayudar? a los agentes de la compa??a con el marketing.
- Tarea 2: predecir si es probable que un nuevo cliente reciba un beneficio de seguro. ?Puede un modelo de predicci?n entrenado funcionar mejor que un modelo dummy no entrenado? - ?Puede funcionar peor? Explica tu respuesta.
- Tarea 3: predecir la cantidad de beneficios de seguro que probablemente recibir? un nuevo cliente utilizando un modelo de regresi?n lineal.
- Tarea 4: proteger los datos personales de los clientes sin romper el modelo de la tarea anterior.

Es necesario desarrollar un algoritmo de transformaci?n de datos que dificulte la recuperaci?n de la informaci?n personal si los datos caen en manos equivocadas. Esto se denomina enmascaramiento de datos u ofuscaci?n de datos. Pero los datos deben protegerse de tal manera que la calidad de los modelos de machine learning no se vea afectada. No es necesario elegir el mejor modelo, basta con demostrar que el algoritmo funciona correctamente.

Descripci?n de datos
El dataset se almacena en el archivo /datasets/insurance_us.csv.

- Caracter?sticas:
    - Gender: Sexo.
    - Age: Edad
    - Salary: Salario
    - Family Members: Numero de Familiares
    - Insurance Benefits: N?mero de familiares de la persona asegurada.
- Objetivo: n?mero de beneficios de seguro recibidos por una persona asegurada en los ?ltimos cinco a?os.

In [ ]:
# Importaci?n de bibliotecas est?ndar
import numpy as np
import pandas as pd

# Importaci?n de bibliotecas de visualizaci?n
import seaborn as sns
import matplotlib.pyplot as plt

# Importaci?n de bibliotecas para machine learning y modelos

from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.linear_model import Ridge
from sklearn.neighbors import NearestNeighbors
from sklearn.linear_model import PoissonRegressor
from sklearn.dummy import DummyClassifier

# T?cnicas de sobremuestreo y submuestreo
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler


#### Carga los datos.


In [ ]:
raw_insurance = pd.read_csv('../datasets/insurance_us.csv')
raw_insurance

In [ ]:
# Verifica que los datos no tengan problemas: no faltan datos, no hay valores extremos, etc.
raw_insurance.info(show_counts=True)

In [ ]:
raw_insurance['Age'] = raw_insurance['Age'].astype('int')
raw_insurance['Salary'] = raw_insurance['Salary'].astype('int')
raw_insurance.info(show_counts=True)

In [ ]:
# Revisamos si tenemos valores duplicados
raw_insurance.duplicated().value_counts()

In [ ]:
# Eliminamos los valores duplicados
raw_insurance.drop_duplicates(inplace=True)

In [ ]:
# Valores At?picos (Outliers) en el salario

# Calculamos los percentiles y el IQR para detectar outliers
Q1 = raw_insurance['Salary'].quantile(0.25)
Q3 = raw_insurance['Salary'].quantile(0.75)
IQR = Q3 - Q1

# Definimos los l?mites superior e inferior para los valores at?picos
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# Creamos un boxplot
plt.figure(figsize=(8,6))
sns.boxplot(data=raw_insurance['Salary'])

# A?adimos l?neas para delimitar los valores at?picos
plt.axhline(y=lower_bound, color='r', linestyle='--', label=f'Lower Bound: {lower_bound:.2f}')
plt.axhline(y=upper_bound, color='g', linestyle='--', label=f'Upper Bound: {upper_bound:.2f}')

# A?adimos t?tulo y etiquetas
plt.title('Distribuci?n de Salarios - Valores At?picos')
plt.xlabel('Caracter?sticas')
plt.ylabel('Salarios')

# Mostramos el gr?fico
plt.legend()
plt.show()

#### Comentarios:

- Solo tenemos verdadera variabilidad en los salarios.
- Tenemos valores atipicos, pero considero que es importante mantenerlos para el objetivo del proyecto.

#### Daremos solucion a cada tarea

In [ ]:
df_insurance = raw_insurance.reset_index(drop=True)
df_insurance

#### Tarea 1: Encontrar clientes que sean similares a un cliente determinado. Esto ayudar? a los agentes de la compa??a con el marketing.

In [ ]:
def find_similar_customers_cosine(df, client_index, top_n=5, weight=2):
    """
    Encuentra los clientes m?s similares a un cliente de referencia utilizando vecinos cercanos con m?trica de distancia coseno.
    
    Par?metros:
    - df: DataFrame con las caracter?sticas escaladas.
    - client_index: ?ndice del cliente de referencia.
    - top_n: N?mero de clientes similares a retornar (excluyendo el cliente de referencia).
    - weight: Factor de ponderaci?n para la columna 'Insurance benefits'.
    
    Retorna:
    - similar_indices: ?ndices de los clientes similares.
    - similar_distances: Distancias coseno correspondientes (menor valor indica mayor similitud).
    """
    # Crear una copia del DataFrame para no modificar el original
    df_weighted = df.copy()
    # Aplicar el peso a la columna "Insurance benefits"
    df_weighted['Insurance benefits'] = df_weighted['Insurance benefits'] * weight
    
    # Instanciar y ajustar el modelo NearestNeighbors con m?trica coseno
    nbrs = NearestNeighbors(n_neighbors=top_n+1, metric='cosine')
    nbrs.fit(df_weighted)
    
    # Obtener las distancias y los ?ndices de los vecinos para el cliente de referencia
    distances, indices = nbrs.kneighbors(df_weighted.iloc[[client_index]])
    
    # El primer vecino es el mismo cliente, por lo que se excluye
    similar_indices = indices[0][1:]
    similar_distances = distances[0][1:]
    
    return similar_indices, similar_distances



In [ ]:
# Cliente similares segun distancia euclidiana.

client_index = 0

similar_customers, distances_to_similar_customers = find_similar_customers_cosine(df_insurance, client_index)

print("Distancias correspondientes:")
print(distances_to_similar_customers , '\n')

print("Clientes de ?ndice: ", client_index )
print(df_insurance.loc[client_index], '\n')

print("Clientes similares al cliente en el ?ndice 0:")
print(df_insurance.loc[similar_customers])

#### Tarea 2: Predecir si es probable que un nuevo cliente reciba un beneficio de seguro. ?Puede un modelo de predicci?n entrenado funcionar mejor que un modelo dummy no entrenado? ?Puede funcionar peor?

In [ ]:
# Revisamos la frecuecia de los registros.

# Calcular el value_counts y ordenar por ?ndice
counts = df_insurance['Insurance benefits'].value_counts().sort_index()

plt.figure(figsize=(10, 6))
bars = plt.bar(counts.index, counts.values, color='skyblue')
plt.yscale('log')  # Establece la escala logar?tmica en el eje Y
plt.xlabel('Insurance benefits')
plt.ylabel('Frecuencia (escala logar?tmica)')
plt.title('Distribuci?n de Insurance benefits (cuadr?cula exponencial)')
plt.grid(True, which="both", linestyle='--', linewidth=0.5)

# Agregar el valor num?rico encima de cada barra
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, height * 1.05, f'{int(height)}',
             ha='center', va='bottom', fontsize=10)

plt.show()


In [ ]:
 # Convertimos a 1 si es mayor a 0 : 1 o mas significa que tiene el beneficio.
df_insurance['Insurance benefits_bin'] = (df_insurance['Insurance benefits'] > 0).astype(int)
df_insurance['Insurance benefits_bin'].value_counts() #Con esto vemos un fuerte desbalance

# Visualizaci?n de la distribuci?n
plt.figure(figsize=(10,4))
plt.bar([0, 1], np.bincount(df_insurance['Insurance benefits_bin']))
plt.title("Comparacion de Target sin Balanceo de Clases")
plt.xlabel("Clase")
plt.ylabel("N?mero de registros")
plt.plot()

In [ ]:
data_insurance = df_insurance.copy(deep=True)

In [ ]:
# Datos de entrenamiento y datos de validacion
features = data_insurance.drop(['Insurance benefits', 'Insurance benefits_bin'], axis=1)
target = data_insurance['Insurance benefits_bin']

features_train, features_valid, target_train, target_valid = train_test_split(features, target, test_size=0.25, random_state=42) # estan ya escalados

In [ ]:
# Estrategia de sobremuestreo: 
# Se usar? SMOTE para aumentar la clase minoritaria
smote = SMOTE(sampling_strategy=0.8, random_state=42)
features_smote, target_smote = smote.fit_resample(features, target)

# Visualizaci?n de la distribuci?n
plt.figure(figsize=(10,4))
plt.subplot(1,2,1)
plt.bar([0, 1], np.bincount(target))
plt.title("Antes de SMOTE")
plt.xlabel("Clase")
plt.ylabel("N?mero de muestras")

plt.subplot(1,2,2)
plt.bar([0, 1], np.bincount(target_smote))
plt.title("Despu?s de SMOTE")
plt.xlabel("Clase")
plt.ylabel("N?mero de muestras")
plt.tight_layout()
plt.show()

# Verificar la distribuci?n de clases despu?s del balanceo
print("Distribuci?n de clases despu?s del balanceo:")
print(pd.Series(target_smote).value_counts())

In [ ]:
# Estrategia de sobremuestreo: 
# Se usar? RandomUnder para disminuir la clase mayoritaria.
under = RandomUnderSampler(sampling_strategy=1, random_state=42)
features_balanced, target_balanced = under.fit_resample(features_smote, target_smote)

# Opcional: Visualizaci?n de la distribuci?n
plt.figure(figsize=(10,4))
plt.subplot(1,2,1)
plt.bar([0, 1], np.bincount(target_smote))
plt.title("Antes de Under")
plt.xlabel("Clase")
plt.ylabel("N?mero de muestras")

plt.subplot(1,2,2)
plt.bar([0, 1], np.bincount(target_balanced))
plt.title("Despu?s de Under")
plt.xlabel("Clase")
plt.ylabel("N?mero de muestras")
plt.tight_layout()
plt.show()

# Verificar la distribuci?n de clases despu?s del balanceo
print("Distribuci?n de clases despu?s del balanceo:")
print(pd.Series(target_balanced).value_counts())

In [ ]:
features_train, features_test, target_train, target_test = train_test_split(features_balanced, target_balanced, random_state=42, test_size=0.75)

In [ ]:
# Escalar solo las caracter?sticas num?ricas
scaler = StandardScaler()
features_train_scaled_2 = scaler.fit_transform(features_train)
features_test_scaled_2 = scaler.transform(features_test)

# Modelo Dummy
dummy_model = DummyClassifier(strategy="most_frequent")
dummy_model.fit(features_train_scaled_2, target_train)
y_pred_dummy = dummy_model.predict(features_test_scaled_2)
dummy_accuracy = accuracy_score(target_test, y_pred_dummy)
print(f'Precisi?n del modelo Dummy: {dummy_accuracy:.2f}')

# Modelo de Regresi?n Log?stica con m?s iteraciones
model = LogisticRegression(max_iter=500, solver='lbfgs')  # Aumentamos max_iter
model.fit(features_train_scaled_2, target_train)
y_pred_model = model.predict(features_test_scaled_2)
model_accuracy = accuracy_score(target_test, y_pred_model)
print(f'Precisi?n del Modelo de Regresi?n Entrenado: {model_accuracy:.2f}')

In [ ]:
# Supongamos que ya tienes calculadas las precisiones:
# dummy_accuracy: Precisi?n del modelo Dummy
# model_accuracy: Precisi?n del modelo de Regresi?n Log?stica


# Crear un diccionario con los nombres de los m?todos y sus precisiones
resultados = {
    'Dummy Classifier': dummy_accuracy,
    'Regresi?n Log?stica': model_accuracy
}

# Extraer las etiquetas y los valores
metodos = list(resultados.keys())
accuracies = list(resultados.values())

# Crear el gr?fico de barras
plt.figure(figsize=(8, 6))
bars = plt.bar(metodos, accuracies, color=['orange', 'blue'])

# Colocar el valor de cada barra encima de la misma
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, height + 0.01, f'{height:.2f}', 
             ha='center', va='bottom', fontsize=12)

plt.ylim(0, 1.1)  # Como la precisi?n est? entre 0 y 1
plt.ylabel('Precisi?n')
plt.title('Comparaci?n de Precisi?n: Dummy vs Regresi?n Log?stica')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()


#### Tarea 3: Predecir la cantidad de beneficios de seguro que probablemente recibir? un nuevo cliente utilizando un modelo de regresi?n lineal.

In [ ]:
# Funci?n para escalar la columna Salary
def scale_salary(features):
    scaler = StandardScaler()
    features = features.copy()  # Evitar modificar el original
    features['Salary'] = scaler.fit_transform(features[['Salary']])
    return features

# Divisi?n de los datos
def split_data(features, target):
    features = scale_salary(features)  # Escalar Salary
    features_train, features_review, target_train, target_review = train_test_split(features, target, random_state=42, test_size=0.3)
    features_test, features_valid, target_test, target_valid = train_test_split(features_review, target_review, random_state=42, test_size=0.6)
    return features_train, features_test, features_valid, target_train, target_test, target_valid

# Entrenamiento de modelos y c?lculo de m?tricas
def train_and_evaluate(features_train, features_test, features_valid, target_train, target_test, target_valid):
    models = {
        "Poisson Regressor": PoissonRegressor(),
        "Linear Regression (Rounded)": LinearRegression(),
        "Random Forest": RandomForestRegressor(n_estimators=20, random_state=42),
        "Logistic Regression": LogisticRegression(solver="lbfgs", max_iter=1000)
    }

    metrics_rmse = {"Model": [], "Set": [], "RMSE": []}
    metrics_r2 = {"Model": [], "Set": [], "R2": []}

    for model_name, model in models.items():
        model.fit(features_train, target_train)

        # Predicciones
        y_train_pred = model.predict(features_train)
        y_test_pred = model.predict(features_test)
        y_valid_pred = model.predict(features_valid)

        # Para regresi?n lineal: redondeamos los valores a enteros
        if model_name == "Linear Regression (Rounded)":
            y_train_pred = np.round(y_train_pred).astype(int)
            y_test_pred = np.round(y_test_pred).astype(int)
            y_valid_pred = np.round(y_valid_pred).astype(int)

        # Calcular RMSE
        rmse_train = np.sqrt(mean_squared_error(target_train, y_train_pred))
        rmse_test = np.sqrt(mean_squared_error(target_test, y_test_pred))
        rmse_valid = np.sqrt(mean_squared_error(target_valid, y_valid_pred))

        # Guardar m?tricas RMSE
        for dataset, rmse in zip(["Entrenamiento", "Test", "Validaci?n"], [rmse_train, rmse_test, rmse_valid]):
            metrics_rmse["Model"].append(model_name)
            metrics_rmse["Set"].append(dataset)
            metrics_rmse["RMSE"].append(rmse)
        
        # Calcular R2
        r2_train = r2_score(target_train, y_train_pred)
        r2_test = r2_score(target_test, y_test_pred)
        r2_valid = r2_score(target_valid, y_valid_pred)

        # Guardar m?tricas R2
        for dataset, r2 in zip(["Entrenamiento", "Test", "Validaci?n"], [r2_train, r2_test, r2_valid]):
            metrics_r2["Model"].append(model_name)
            metrics_r2["Set"].append(dataset)
            metrics_r2["R2"].append(r2)

    return [pd.DataFrame(metrics_rmse), pd.DataFrame(metrics_r2)]

# Llamado a las funciones
features_train, features_test, features_valid, target_train, target_test, target_valid = split_data(features_balanced, target_balanced)
metrics_df = train_and_evaluate(features_train, features_test, features_valid, target_train, target_test, target_valid)

# Gr?ficos de comparaci?n con etiquetas
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Gr?fico de RMSE
sns.barplot(ax=axes[0], data=metrics_df[0], x="Set", y="RMSE", hue="Model", dodge=True)
axes[0].set_title("Comparaci?n de Modelos - RMSE")
axes[0].set_xlabel("Conjunto de Datos")
axes[0].set_ylabel("RMSE")
axes[0].legend(title="Modelo", loc="upper center", bbox_to_anchor=(0.5, -0.1), ncol=2)
for container in axes[0].containers:
    axes[0].bar_label(container, fmt="%.2f", label_type="edge", fontsize=10)

# Gr?fico de R?
sns.barplot(ax=axes[1], data=metrics_df[1], x="Set", y="R2", hue="Model", dodge=True)
axes[1].set_title("Comparaci?n de Modelos - R?")
axes[1].set_xlabel("Conjunto de Datos")
axes[1].set_ylabel("R?")
axes[1].legend(title="Modelo", loc="upper center", bbox_to_anchor=(0.5, -0.1), ncol=2)
for container in axes[1].containers:
    axes[1].bar_label(container, fmt="%.2f", label_type="edge", fontsize=10)

plt.tight_layout()
plt.show()


CONCLUSIONES:

Tarea 3: 
- Para este ejercicio tomamos los datos y escalamos solo los valores de salary ya que presenta enorme variabilidad
- Usando una Poison Regressor tenemos:
    - Entrenamiento: 
        - RMSE = 0.36
        - R2 = 0.47 
    - Prueba: 
        - RMSE = 0.43
        - R2 = 0.25
    - Validacion: 
        - RMSE = 0.35
        - R2 = 0.50
- Usando un Linear Regressor (Rounded) tenemos:
    - Entrenamiento: 
        - RMSE = 0.24
        - R2 = 0.77 
    - Prueba: 
        - RMSE = 0.25
        - R2 = 0.75
    - Validacion: 
        - RMSE = 0.24
        - R2 = 0.75
- Usando un Random Forest Regressor tenemos:
    - Entrenamiento: 
        - RMSE = 0.01
        - R2 = 1
    - Prueba: 
        - RMSE = 0.02
        - R2 = 1
    - Validacion: 
        - RMSE = 0.01
        - R2 = 1
- Usando un Logistic Regression tenemos:
    - Entrenamiento: 
        - RMSE = 0.05
        - R2 = 0.99
    - Prueba: 
        - RMSE = 0.03
        - R2 = 1
    - Validacion: 
        - RMSE = 0.04
        - R2 = 0.99
- Tenemos que Random Forest Regressor tiene mejores metricas con entrenamiento, test y validacion.


#### Tarea 4: Proteger los datos personales de los clientes sin romper el modelo de la tarea anterior.

In [ ]:
#Copiamos el dataframe a otra variable para esta tarea
insurance = df_insurance.copy(deep=True)
insurance

In [ ]:
# Consideraremos que edad 'Age' y salario 'Salary' como datos sensibles

# 1. Enmascarar 'age' (generalizaci?n en rangos)
bins = [18, 25, 35, 45, 55, 60, 65]
labels = ['18-25', '26-35', '36-45', '46-55', '56-60', '+61']

insurance['age_group'] = pd.cut(insurance['Age'], bins=bins, labels=labels)
insurance = insurance.drop(columns=['Age'], axis=1)  # Eliminamos la edad exacta

# 2. Enmascarar 'salary' a?adiendo ruido controlado

noise_std = 0.05 * insurance['Salary'].std()
noise = np.random.normal(0, noise_std, insurance['Salary'].shape)  # Ruido con desviaci?n est?ndar 2000
insurance['salary_masked'] = insurance['Salary'] + noise
insurance = insurance.drop(columns=['Salary'],axis=1)  # Eliminamos el salario original
insurance


In [ ]:
# Codificamos la columna 'age_group' usando One Hot Encoding

# Usamos OneHotEncoder para transformar la columna 'age_group'
# El par?metro sparse_output=False nos asegura que la salida ser? un array denso (no una matriz dispersa)
encoder = OneHotEncoder(sparse_output=False)  # Cambi? 'sparse=False' por 'sparse_output=False' para las versiones m?s recientes de sklearn
age_group_encoded = encoder.fit_transform(insurance[['age_group']])

# Convertimos el array resultante en un DataFrame
# Utilizamos get_feature_names_out para obtener los nombres de las columnas generadas por la codificaci?n
age_group_encoded_insurance = pd.DataFrame(age_group_encoded, columns=encoder.get_feature_names_out(['age_group']))

# Concatenamos las nuevas columnas codificadas con el DataFrame original
insurance = pd.concat([insurance, age_group_encoded_insurance], axis=1)

# Eliminamos la columna original 'age_group' que ya ha sido codificada
insurance = insurance.drop(columns=['age_group'], axis=1)

# Mostramos el DataFrame final con las columnas codificadas
print(insurance.head())  # Para verificar que la codificaci?n fue exitosa


In [ ]:
# Dividimos datos en caracteristicas y objetivo
features_4 = insurance.drop(columns=['Insurance benefits'])  # 'Insurance benefits' es el target
# Dividimos datos en caracteristicas y objetivo
target_4 = insurance['Insurance benefits']

features_train, features_valid, target_train, target_valid = train_test_split(features_4, target_4, test_size=0.25, random_state=42)

In [ ]:
# Entrenamos el modelo de regresi?n lineal

# Definimos el modelo de regresi?n lineal
model = LinearRegression()

# Ajustamos el modelo a los datos de entrenamiento
model.fit(features_train, target_train)

# Realizamos las predicciones sobre el conjunto de validaci?n
y_pred = model.predict(features_valid)

# Evaluamos el rendimiento del modelo calculando el MSE y el R2
mse = mean_squared_error(target_valid, y_pred)  # Calculamos el error cuadr?tico medio
r2 = r2_score(target_valid, y_pred)  # Calculamos el coeficiente de determinaci?n R2

# Imprimimos las m?tricas de evaluaci?n
print(f'MSE (Error cuadr?tico medio): {mse:.2f}')
print(f'R2 (Coeficiente de determinaci?n): {r2:.2f}')


In [ ]:
# Entrenamos el modelo RandomForestRegressor con los datos ofuscados

# Definimos el modelo RandomForestRegressor con un valor fijo para random_state para asegurar reproducibilidad
model = RandomForestRegressor(random_state=42)

# Ajustamos el modelo con los datos de entrenamiento
model.fit(features_train, target_train)

# Realizamos las predicciones sobre el conjunto de validaci?n
y_pred = model.predict(features_valid)

# Evaluamos el rendimiento del modelo calculando el MSE (Error cuadr?tico medio) y el R2 (Coeficiente de determinaci?n)
mse = mean_squared_error(target_valid, y_pred)  # C?lculo del error cuadr?tico medio
r2 = r2_score(target_valid, y_pred)  # C?lculo del coeficiente de determinaci?n

# Imprimimos los resultados de las m?tricas de evaluaci?n
print(f'MSE (Error cuadr?tico medio): {mse:.2f}')
print(f'R2 (Coeficiente de determinaci?n): {r2:.2f}')

### 4. Saca conclusiones basadas en tu experiencia trabajando en el proyecto.

#### CONCLUSIONES
Tarea 1: B?squeda de clientes similares
- Utilizamos el algoritmo Nearest Neighbors con la m?trica 'cosine', lo que nos permite medir la similitud en funci?n del ?ngulo entre los vectores de caracter?sticas. Esto es ?til para encontrar clientes con perfiles similares.
- Implementamos un algoritmo que identifica los 5 clientes m?s similares a un perfil dado en funci?n de esta m?trica.

Tarea 2: Predicci?n de clientes que recibir?n beneficios
- Aplicamos Regresi?n Log?stica para identificar a los clientes con mayor probabilidad de recibir un beneficio.
- Transformamos la variable objetivo Insurance benefits en una variable binaria, donde todos los valores mayores a 0 se convirtieron en 1.
- Realizamos escalamiento de datos para optimizar el procesamiento del modelo.
- Aplicamos t?cnicas de balanceo de clases, ya que los datos estaban desbalanceados.
- Comparamos los resultados de la Regresi?n Log?stica con un modelo base (dummy) que asigna etiquetas aleatorias con una precisi?n de 0.5. Nuestro modelo de Regresi?n - - 
- Log?stica alcanz? una precisi?n del 99%, demostrando su efectividad.

Tarea 3: Comparaci?n de modelos para predicci?n de Salary
- Escalamos ?nicamente la variable Salary, ya que presentaba una gran variabilidad en comparaci?n con las dem?s caracter?sticas.
- Probamos cuatro modelos de regresi?n para determinar el m?s ?ptimo:
- Random Forest Regressor result? ser el modelo con mejor desempe?o, obteniendo los siguientes resultados:
- Entrenamiento:
    - RMSE = 0.01
    - R? = 1.00
- Prueba:
    - RMSE = 0.02
    - R? = 1.00
- Validaci?n:
    - RMSE = 0.01
     - R? = 1.00
- Estos resultados sugieren que el modelo comprende bien las relaciones entre las caracter?sticas y generaliza correctamente. Sin embargo, dado que el max_depth del modelo fue 1, podr?a existir un riesgo de sobreajuste. No obstante, al comparar los resultados en los conjuntos de prueba y validaci?n, observamos un rendimiento estable, lo que indica que el modelo generaliza adecuadamente.

Tarea 4: Ofuscaci?n de datos y evaluaci?n de impacto
- Aplicamos un m?todo de ofuscaci?n para proteger la informaci?n sensible de los clientes:
- Agrupamos la variable Age en rangos de edad.
- Introdujimos ruido controlado en la variable Salary.
- Entrenamos nuevos modelos con los datos enmascarados y utilizamos One-Hot Encoding para procesar las variables categ?ricas.
- Evaluamos el impacto de la ofuscaci?n comparando los resultados con los datos originales:
- Linear Regression:
    - MSE: 0.02
    - R?: 0.92
- Random Forest Regressor:
    - MSE: 0.03
    - R?: 0.88

- Linear Regression mostr? m?tricas m?s estables despu?s de la ofuscaci?n, lo que indica que pudo adaptarse mejor a los datos transformados.
Random Forest Regressor, en cambio, se vio m?s afectado, ya que este modelo depende de relaciones m?s complejas, las cuales se redujeron debido a la ofuscaci?n.